# IoTID20 — 2026 Paper-Inspired Reproduction & DevOps Model Export

**Research paper:** Zaki et al., *Attention-Enhanced Hybrid Deep Learning for Intrusion Detection in IoT Networks*, *IET Networks* (2026), DOI [10.1049/ntw2.70037](https://doi.org/10.1049/ntw2.70037). **Dataset:** [officially cited Kaggle IoTID20 dataset](https://www.kaggle.com/datasets/rohulaminlabid/iotid20-dataset).

**IMPORTANT:** Notebook cells are **unexecuted** until you run them with original data. No numbers below are claimed as reproduced. The paper reports 1.0000 binary, 0.9995 5-class and 0.8228/0.8225 9-class accuracy. Source code is available only upon request, so this is an approximation rather than a verified independent exact implementation. The paper includes ADASYN vs ProWRAS; this notebook implements an ADASYN-based experimental branch, but not ProWRAS.

Paper reports 83 candidate features after removing `Label`, `Cat`, `Sub_Cat` from 86 raw columns, with `SelectKBest(k=70)`, 80/20 stratified split, scaling, features reshaped to (70,1), Nadam (0.001), batch 512, max 25 epochs, 10% validation and early stopping. Paper row counts differ from some publicly circulated copies; log your actual count.

## 1. Imports and deterministic setup

In [ ]:
import os, sys, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import joblib
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SEED=42
np.random.seed(SEED);random.seed(SEED)
print('Project root:',ROOT)

## 2. Download original dataset (once; requires internet access)

In [ ]:
import subprocess
raw_dir=ROOT/'data'/'raw'
if not list(raw_dir.rglob('*.csv')) if raw_dir.exists() else True:
    subprocess.check_call([sys.executable,str(ROOT/'scripts/download_dataset.py')])
print('CSV files:', [str(p.relative_to(ROOT)) for p in raw_dir.rglob('*.csv')])

## 3. Load, inspect exact targets and class counts

In [ ]:
paths=list(raw_dir.rglob('*.csv'))
frames=[]
for p in paths:
    d=pd.read_csv(p,low_memory=False)
    d.columns=[str(c).strip() for c in d.columns]
    if 'Label' in d.columns: frames.append((p,d))
assert frames,'No CSV with Label column found. Inspect original Kaggle file.'
path,df=max(frames,key=lambda z:len(z[1]))
print('Selected:',path,'shape:',df.shape)
print(df[['Label']].value_counts(dropna=False).head(30))
print('All columns:',df.columns.tolist())

## 4. Prepare leakage-free binary train/test sets (do not use Cat/Sub_Cat as inputs)

In [ ]:
df=df.replace([np.inf,-np.inf],np.nan).dropna(subset=['Label'])
exclude=[c for c in df if c.casefold() in ('label','cat','sub_cat')]
X=df.drop(columns=exclude).dropna(axis=1,how='all').copy()
for c in X.select_dtypes(include='object'):
    converted=pd.to_numeric(X[c],errors='coerce')
    if converted.notna().mean()>.98: X[c]=converted
le=LabelEncoder(); y=le.fit_transform(df['Label'].astype(str).str.strip())
assert len(le.classes_)==2, f'Unexpected binary target labels: {le.classes_}'
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=SEED,stratify=y)
print('Records:',len(df),'raw predictors:',X.shape[1],'train:',len(Xtr),'test:',len(Xte),'classes:',list(le.classes_))

## 5. Fit preprocessing on training data only, select up to 70 encoded features

In [ ]:
numeric=X.select_dtypes(include='number').columns.tolist()
categorical=[c for c in X if c not in numeric]
preprocess=ColumnTransformer([
    ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
    ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical)],sparse_threshold=0)
Ztr=preprocess.fit_transform(Xtr);Zte=preprocess.transform(Xte)
selector=SelectKBest(f_classif,k=min(70,Ztr.shape[1]))
Ztr=selector.fit_transform(Ztr,ytr).astype('float32')
Zte=selector.transform(Zte).astype('float32')
print('Selected feature count:',Ztr.shape[1])

## 6. Baseline comparison: Random Forest

In [ ]:
rf=RandomForestClassifier(n_estimators=150,random_state=SEED,n_jobs=-1)
rf.fit(Ztr,ytr);p_rf=rf.predict(Zte)
print('RF accuracy:',accuracy_score(yte,p_rf),'weighted F1:',f1_score(yte,p_rf,average='weighted'))
print(classification_report(yte,p_rf,target_names=list(le.classes_)))

## 7. Experimental CNN–GRU with learned branch-level attention (ADASYN)

This **paper-inspired approximation** trains two parallel branches on each selected feature vector reshaped into a pseudo-sequence of 70 values. Exact paper architecture and ProWRAS source implementation are not publicly provided. ADASYN oversamples *training only*; test remains untouched. Running requires `tensorflow` and `imbalanced-learn` on a machine with sufficient RAM/GPU.

In [ ]:
from imblearn.over_sampling import ADASYN
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
tf.random.set_seed(SEED)
ada=ADASYN(random_state=SEED,n_neighbors=5)
X_bal,y_bal=ada.fit_resample(Ztr,ytr)
print('Balanced training:',X_bal.shape, pd.Series(y_bal).value_counts().to_dict())
X_bal=X_bal.reshape((-1,Ztr.shape[1],1)).astype('float32')
X_test_seq=Zte.reshape((-1,Zte.shape[1],1)).astype('float32')

In [ ]:
nfeat=Ztr.shape[1];n_classes=len(le.classes_)
inp=keras.Input(shape=(nfeat,1),name='feature_pseudo_sequence')
# Parallel convolutional and recurrent branches; not a claim of exact author code.
a=layers.Conv1D(64,3,padding='same',activation='relu')(inp)
a=layers.Dropout(.2)(a)
a=layers.Conv1D(64,3,padding='same',activation='relu')(a)
a=layers.Dropout(.3)(a)
a=layers.Conv1D(32,3,padding='same',activation='relu')(a)
conv_vec=layers.GlobalAveragePooling1D()(a)
b=layers.GRU(64,return_sequences=True)(inp)
b=layers.GRU(32)(b)
conv_embed=layers.Dense(64,activation='relu')(conv_vec)
gru_embed=layers.Dense(64,activation='relu')(b)
stack=layers.Lambda(lambda tensors: tf.stack(tensors,axis=1))([conv_embed,gru_embed])
weights=layers.Dense(1)(layers.Activation('tanh')(stack))
weights=layers.Softmax(axis=1,name='branch_attention')(weights)
weighted=layers.Multiply()([stack,weights])
combined=layers.GlobalAveragePooling1D()(weighted)
combined=layers.Dropout(.4)(combined)
out=layers.Dense(n_classes,activation='softmax')(combined)
net=keras.Model(inp,out)
net.compile(optimizer=keras.optimizers.Nadam(learning_rate=.001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
net.summary()

In [ ]:
callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',patience=6,restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=4,factor=.5,min_lr=1e-6)]
history=net.fit(X_bal,y_bal,epochs=25,batch_size=512,validation_split=.1,callbacks=callbacks,verbose=1)
pred=np.argmax(net.predict(X_test_seq,batch_size=512),axis=1)
actual={'accuracy':float(accuracy_score(yte,pred)),
        'weighted_f1':float(f1_score(yte,pred,average='weighted')),
        'macro_f1':float(f1_score(yte,pred,average='macro'))}
print('Paper binary accuracy: 1.0000; paper-inspired independent result:',actual)
print('Confusion matrix:\n',confusion_matrix(yte,pred))
print(classification_report(yte,pred,target_names=list(le.classes_)))
(ROOT/'docs'/'paper_inspired_metrics.json').write_text(json.dumps(actual,indent=2))

## 8. Export a deployable fitted scikit-learn model

The Flask app intentionally deploys the **Random Forest baseline**, not the Keras experimental network. Exporting a fully equivalent Keras inference adapter with exact preprocessing would be a separate model-specific deployment effort (see manual Stage 4.4). This prevents mislabelling the baseline as the paper model.

In [ ]:
subprocess.check_call([sys.executable,str(ROOT/'scripts/train_model.py')])
print('Trusted fitted baseline:',ROOT/'backend/models/model_bundle.joblib')
print('Local scores:',(ROOT/'docs/actual_metrics.json').read_text()[:1200])

## 9. Reproduction verdict

Compare `docs/paper_inspired_metrics.json` to published 1.0000 binary test accuracy, and record the different row counts, model and oversampling choices, TensorFlow version, hardware and seed. Do not claim 2026 paper reproduction unless the exact author pipeline is reconstructed and independently matches evaluation protocol. Also evaluate **five-class** (`Cat`) and **nine-class** (`Sub_Cat`) separately if those claims are needed; they are **not reproduced** here.